# Presentation notes: MS_DSA vs lysine structural features

Notes for reviewing the design, the aggregation choices and the sanity checks before presenting, organized as the
questions another scientist is likely to ask. Numbers quoted in the text come from the analyses run while building
the pipeline; the last section recomputes the main ones from the current files.

**Contents**
0. Terminology
1. Pipeline at a glance
2. MS_DSA: definition, versions and flags
3. Peptide to lysine mapping
4. Structures: which models, what is kept
5. Solvent accessibility (FreeSASA)
6. Keeping chain identity through FreeSASA
7. Contacts: definition and classes
8. Aggregation: copies, lysines, peptides
9. No contact: definition, exhaustiveness, overlaps, coverage
10. Monomer vs multimer: what is (never) mixed
11. B-factors
12. Plots and statistics
13. Sanity checks done (question → answer)
14. Anticipated questions
15. Known limitations and open items
16. Live checks of key numbers

## 0. Terminology

### MS side
| term | meaning |
|---|---|
| **run** | one MS acquisition (`Run.Index` 0–4); everything MS-related is computed per run |
| **light / heavy** | dimethyl label channels: light = `Channel '0'`, heavy = `Channel '8'` |
| **`Ms1.Area`** | MS1 peak area of a precursor in one channel; summed to get the light and heavy areas |
| **precursor / charge state** | a peptide at one charge (2, 3 or 4 here; no charge 1) |
| **peptide-run group** | one (peptide, run); MS_DSA is one value per group |
| **MS_DSA** | `light / (light + heavy)`, all charge states of each channel summed (the original definition) |
| **shared charges / `MS_DSA_shared_charges`** | MS_DSA from only the charge states measured in **both** channels; NaN if a channel is absent or no charge is shared |
| **`channel_absent` / `absent_channel`** | no row at all for one channel (`light` or `heavy`) |
| **`no_shared_charges`** | both channels present, but no charge state in common |
| **`zero_area` / `zero_area_shared`** | both channels present, but one channel's summed area (all / shared charges) is 0; `…_channel` says which (`light`, `heavy`, `both`) |
| **`multiple_charges`** | more than one charge state in a channel |
| **`peptide_mapped`** | the peptide was found in its UniProt sequence |
| **version A** | shared-charge MS_DSA, **zero areas included**: excludes `channel_absent`, `no_shared_charges` (MS_DSA can be 0 or 1) |
| **version B** | shared-charge MS_DSA, **zero areas excluded**: also excludes `zero_area_shared` (0 < MS_DSA < 1) |
| **all charges vs shared charges** | the two MS_DSA computations above |

### Structures and chains
| term | meaning |
|---|---|
| **PDB / multimer** | experimental structures, used as their **biological assembly**; SASA of the whole complex |
| **AlphaFold / monomer / AF** | the predicted single-chain model of the protein |
| **biological assembly** | the annotated functional complex (first assembly used); built from the deposited model |
| **asymmetric unit / deposited model** | the coordinates as deposited; B-factors come from here |
| **bio chain** | a chain of the assembly, e.g. `A1`, `A2` (copies of author chain `A`) |
| **author chain** | the chain id in the deposited file, e.g. `A` |
| **chain copy / copy** | one occurrence of a lysine in one bio chain of one structure; a lysine has many copies across structures and assembly copies |
| **entity** | a distinct molecule in the file (one entity = one UniProt protein for chains that map) |
| **PDBe / SIFTS mapping** | residue-level PDB → UniProt mappings (PDBe for 37,674 structures, SIFTS for 10) |
| **residue label** | the unique number (+ insertion code) given to every residue before FreeSASA, so results map back to the right bio chain |
| **covered** | a lysine appears (with an NZ) in that source; uncovered lysines have NA flags |

### Accessibility and B-factors
| term | meaning |
|---|---|
| **`asa_nz`** | solvent-accessible surface area of the lysine NZ atom (Å²), FreeSASA, ProtOr radii, Lee–Richards |
| **`rel_asa`** | relative residue SASA (fraction of the reference maximum) |
| **monomer `asa_nz`** | from AlphaFold: `asa_nz`, and per peptide `asa_nz_{avg,min,max}` |
| **`asa_nz_multimer_*`** | multimer `asa_nz` over **all** chain copies of the lysine (older, class-independent) |
| **`asa_nz_pdb_{inter\|intra}_{class}_*`** | multimer `asa_nz` over only the copies **with that contact** |
| **`n_pdb_{inter\|intra}_{class}`** | number of such copies that have an `asa_nz` |
| **B-factor z-score** | B-factor normalized over the protein heavy atoms of its chain (0 = chain average) |
| **`nz_bfactor_z` / `avg_bfactor_z`** | NZ atom / residue-average B-factor z-score |

### Contacts and groups
| term | meaning |
|---|---|
| **contact** | lysine NZ within 6.5 Å of an atom of a non-adjacent protein or nucleic-acid residue (stored contacts) |
| **inter-chain / intra-chain (contact type)** | the partner is on a different / the same bio chain (homo-oligomer copies count as inter) |
| **contact class / class** | a criterion a contact must satisfy: `dist4.5`, `salt_bridge`, `hydrogen_bond`, `cation_pi`, `covalent_isopeptide`, `any_bond` |
| **`dist4.5`** | any contact with the NZ within 4.5 Å (distance-based definition) |
| **`any_bond`** | a contact satisfying any of the four bond types |
| **`assume_lysine`** | the bond-detection method used for all bond types |
| **`pdb_inter_{class}` / `pdb_intra_{class}` / `af_intra_{class}`** | flags: some copy of the lysine (peptide: some lysine) has that contact |
| **no contact (`no_contact_pdb`, `no_contact_af`)** | no contact within 4.5 Å in any copy of that source; for a peptide: all lysines |
| **purely intra (`purely_intra_pdb`, `purely_intra_af`)** | an intra-chain class and no lysine with an inter-chain bond (`pdb_inter_any_bond`) |
| **Inter PDB, Intra PDB, Intra AF, No contact PDB, No contact AF** | the plot groups (membership + value from one source each; Inter always PDB multimer) |
| **"intra may also be inter" / "purely intra"** | the two comparisons: intra groups may or may not include peptides that are also inter-chain |
| **sasa_source "monomer" / "multimer"** | eval7 setting: intra and no-contact groups from AlphaFold or from PDB |
| **old flags: `inter_chain_pdb`, `intra_chain_pdb`, `intra_chain_alphafold`, `{bond}_pdb`** | earlier per-lysine flags (any bond type; bond flags regardless of contact type); `pdb_inter_any_bond` equals `inter_chain_pdb` |
| **strict (old)** | eval2–eval5 option: inter-chain peptides without an intra-chain PDB contact |
| **Not Contact (old)** | eval2–eval5 group: no contact of the chosen bond type |

### Aggregation
| term | meaning |
|---|---|
| **max-of-max (`…_max_max`)** | max over the copies, then max over the peptide's lysines |
| **min-of-min (`…_min_min`)** | min over the copies, then min over the lysines |
| **average (`…_mean_avg`)** | mean over the copies of each lysine, then mean over the lysines |
| **all lysines required** | older columns: a peptide value is NaN unless every lysine has a value |
| **lysines in the class** | contact-class columns: aggregated over the lysines that are in the class only |

### Data products and code
| term | meaning |
|---|---|
| **lysine table** | `evaluation1/lysine_properties.parquet`, one row per UniProt lysine |
| **peptide-lysine table** | `evaluation1/peptide_lysines.parquet`, one row per (peptide, lysine) |
| **run tables** | `evaluation1/runs/run_{0..4}.parquet`, one annotated MS_DSA table per run |
| **`MSDSAAnnotator`, `LysinePropertyTable`** | map peptides to lysines and aggregate / build the lysine table |
| **`SASAMonomerStore`, `SASAMultimerStore`, `LysineBFactorStore`** | saved per-protein / per-structure SASA and B-factor objects |
| **`ComputeManager`** | runs a compute function over many keys, saving results to a store (parallel workers) |
| **pooled** | all runs put together (scatter plots, correlations, Venn diagrams) |
| **run-averaged histogram** | per run, percent within each group per bin; then the mean over runs (error bars: SD over runs) |
| **percent within group** | each group's bars sum to 100%, so groups of different sizes compare |
| **`require=`** | only count peptides with a value in the given MS_DSA column |

### Meanings that changed (retired notebooks in `obsolete/`)
| term | where | meaning |
|---|---|---|
| **section 1 / 2 / 3** | eval3 | all peptides (all-charge MS_DSA) / shared, strict / both channels observed |
| **A / B / C** | eval4 | A and B as above; **C** = intra and no-contact groups also from multimer (all copies) |

## 1. Pipeline at a glance

```
report.parquet (DIA-NN, light '0' / heavy '8' dimethyl channels, 5 runs)
  └─ ms_dsa_v2.py            MS_DSA per (peptide, run) + flags          → data/ms_data/report_ms_dsa_v2.csv
       └─ MSDSAAnnotator      peptide → UniProt lysine positions
            └─ LysinePropertyTable  per-lysine structural features     → evaluation1/lysine_properties.parquet
                 ├─ contacts:   ProteinContacts (PDB, AlphaFold)          binding_interfaces/…
                 ├─ SASA:       SASAMonomer (AlphaFold), SASAMultimer (PDB assemblies)   sasa/…
                 └─ B-factors:  LysineBFactor (PDB)                        bfactor/…
            └─ peptide aggregation per run                              → evaluation1/runs/run_{0..4}.parquet
                 └─ notebooks eval6, eval7_monomer, eval7_multimer, eval8_venn (plots.py)
```

**Intuition.** MS_DSA measures how much of a peptide's lysine(s) was labelled in the accessible (light) state.
We compare it with how accessible the lysine's NZ is in structures, and whether the lysine sits at an inter-chain
interface, an intra-chain contact, or nowhere near anything. The question: does labelling track structural
accessibility and interface burial?

## 2. MS_DSA: definition, versions and flags

**Definition.** Per (peptide, run): `MS_DSA = light / (light + heavy)`, light and heavy being summed `Ms1.Area`
(channel `'0'` light, `'8'` heavy). **Computed separately for each run.** Only lysine-containing peptides are kept
(the rows with an empty channel are all non-lysine peptides).

**Two computations**
- `MS_DSA`: all charge states of each channel summed (the original definition).
- `MS_DSA_shared_charges`: only charge states measured in **both** channels. Not computed (NaN) if a channel is
  absent or no charge state is shared.

**Why shared charges?** Light was often seen at more charge states than heavy (1,482 peptide-runs with more light
charge states vs 15 with more heavy). Summing all charges then inflates light: in clean groups the all-charge
MS_DSA is higher by +0.036 on average (58 groups shifted by > 0.1). There are no charge-1 precursors (only 2, 3, 4),
so "charge 1 only" was not possible.

**Flags (never used to drop rows silently; plots filter explicitly)**
| flag | meaning |
|---|---|
| `channel_absent` / `absent_channel` | no row at all for light or heavy |
| `no_shared_charges` | both channels have rows but no charge state in common |
| `zero_area` / `zero_area_channel` | both channels have rows, but a channel's summed area is 0 |
| `zero_area_shared` / `zero_area_channel_shared` | same for the shared-charge areas |

**How common these are** (54,760 peptide-run groups): one channel absent **86.4%** (mostly heavy absent: 44,319
vs 2,986); both present but a zero area 7.7%; both present and both > 0 only **5.9%**.
A missing channel is not evidence of 0 or 1 accessibility — MS data are missing at random — hence the colleague's
advice to keep these separate.

**Versions used in the plots**
- **A: zero areas included**: exclude `channel_absent`, `no_shared_charges` (MS_DSA may be exactly 0 or 1).
- **B: zero areas excluded**: also exclude `zero_area_shared` (0 < MS_DSA < 1).

Per run, about 1,350–1,850 peptides are in A and 670–780 in B.

## 3. Peptide to lysine mapping

- `Protein.Ids` with several accessions are split into one row per UniProt id (a shared peptide is analysed for
  each protein).
- A peptide is placed at its **first exact match** in the UniProt sequence; every K of the peptide becomes a
  lysine `(uniprot_id, unp_resnum)`. Peptides not found are flagged `peptide_mapped = False` and excluded (2 in run 0).
- A peptide can contain **several lysines**; every structural property is first computed per lysine, then
  aggregated over the peptide's lysines (section 8).

### How many peptides have more than one lysine?

Why it matters: for a multi-lysine peptide MS_DSA is one number for all its lysines, while the structural features
are per lysine and have to be aggregated (max / min / avg, section 8); a peptide joins a contact group if **any** of
its lysines is in it. The more multi-lysine peptides, the more these aggregation choices matter.

Counted on the mapped peptides of the annotated run tables (one row per peptide and UniProt id); "unique" pools the
five runs, "per run" counts each run separately.

In [1]:
import pandas as pd
from dsa.ms_dsa.evaluation1.ms_dsa_annotator import MSDSAAnnotator

run_tables = {r: MSDSAAnnotator.load_run(r) for r in range(5)}
subsets = {
    "all lysine peptides": [],
    "shared-charge MS_DSA, version A (zero areas included)": ["channel_absent", "no_shared_charges"],
    "shared-charge MS_DSA, version B (zero areas excluded)": ["channel_absent", "no_shared_charges", "zero_area_shared"],
}

def select(df, exclude, need_dsa):
    keep = df["peptide_mapped"].copy()
    if need_dsa:
        keep &= df["MS_DSA_shared_charges"].notna()
    for flag in exclude:
        keep &= ~df[flag]
    return df[keep]

def lysine_distribution(n_lysines):
    counts = n_lysines.clip(upper=4).value_counts().sort_index()
    row = {("1 K" if k == 1 else f"{k} K" if k < 4 else "4+ K"): v for k, v in counts.items()}
    row["n"] = len(n_lysines)
    row["% with > 1 K"] = round(100 * (n_lysines > 1).mean(), 1)
    return row

unique_rows, per_run_rows = {}, {}
for name, exclude in subsets.items():
    need_dsa = name != "all lysine peptides"
    selected = {r: select(df, exclude, need_dsa) for r, df in run_tables.items()}
    pooled = pd.concat(selected.values()).drop_duplicates(["Uniprot", "peptide_seq"])
    unique_rows[name] = lysine_distribution(pooled["n_lysines"])
    per_run_rows[name] = {f"run {r}": round(100 * (df["n_lysines"] > 1).mean(), 1) for r, df in selected.items()}

print("Unique peptides pooled over runs: number of lysines per peptide")
display(pd.DataFrame(unique_rows).T.fillna(0).astype({"n": int}))
print("Per run: % of peptides with more than one lysine")
display(pd.DataFrame(per_run_rows).T)

Unique peptides pooled over runs: number of lysines per peptide


,1 K,2 K,3 K,4+ K,n,% with > 1 K
all lysine peptides,12953.0,5333.0,1345.0,246.0,19877,34.8
"shared-charge MS_DSA, version A (zero areas included)",2277.0,416.0,56.0,7.0,2756,17.4
"shared-charge MS_DSA, version B (zero areas excluded)",1452.0,208.0,17.0,3.0,1680,13.6


Per run: % of peptides with more than one lysine


,run 0,run 1,run 2,run 3,run 4
all lysine peptides,35.0,32.6,36.3,34.4,33.1
"shared-charge MS_DSA, version A (zero areas included)",17.3,15.6,17.3,15.9,17.1
"shared-charge MS_DSA, version B (zero areas excluded)",11.5,11.2,11.1,11.4,12.1


**Observation.** About 35% of all lysine peptides have more than one lysine, but only ~17% of the peptides with a
shared-charge MS_DSA (version A) and ~14% in version B: multi-lysine peptides are under-represented among peptides
quantified in both channels. A possible explanation (not verified here): with two or more lysines a peptide can also
occur with mixed labels (some lysines light, some heavy), which splits its signal and may not be reported as channel
`'0'` or `'8'`, so both pure channels are less often observed. Worth checking how DIA-NN's channels treat
multi-lysine peptides before presenting this.

## 4. Structures: which models, what is kept

**PDB (multimer).** For each structure mapped to a protein, the **first biological assembly** (`gemmi.make_assembly`,
chain copies named `A1`, `A2`, …) — the same assembly the contacts are computed on. If no assembly is annotated,
the deposited model is used. 37,684 structures (the contacts set); 37,683 have SASA.

**What is kept for SASA:** protein residues only, including modified amino acids (e.g. MSE, written as ATOM so
FreeSASA keeps them). Removed: nucleic acids, ligands, cofactors, glycans, water, hydrogens, alternate conformations.
So burial by DNA/RNA or ligands does not lower `asa_nz`.

**AlphaFold (monomer).** One predicted model per protein; single chain, so no inter-chain information.

**UniProt mapping of PDB residues.** PDBe residue mappings (37,674 structures) or SIFTS (10). A chain is accepted
only if ≤ 10% of mapped residues mismatch the UniProt sequence (and > 10 residues, < 20 mismatches); mismatching
residues are not mapped. Consequence: some chains (allelic variants, engineered constructs, e.g. HLA-A2 in 3d3v,
TCR chains) are unmapped, so their lysines never enter the analysis.

## 5. Solvent accessibility (FreeSASA)

- **Monomer (AlphaFold):** FreeSASA Python bindings, ProtOr radii, Lee–Richards. One value per residue.
- **Multimer (PDB):** FreeSASA command line (`freesasa --cif --radii=protor --format=json --depth=atom`),
  Lee–Richards, probe 1.4 Å, 20 slices, on the **whole assembly**: burial by partner chains lowers the value.
- **`asa_nz`** is the SASA of the lysine NZ atom (Å²), the atom that gets labelled. Relative areas are stored as
  fractions (FreeSASA JSON reports percent).

**Subtleties**
- Lysines whose side chain stops before NZ (common in low-resolution cryo-EM, e.g. 8vrj) have no `asa_nz`.
- Typical exposed NZ ≈ 55–59 Å². Only **16 of 25,743** lysines exceed 60 Å² in some PDB copy: 15 between 60 and 65
  (stretched CE–NZ bond, 1.50–1.73 Å vs the usual ~1.49 Å, so CE covers less of NZ) and one at 81.4 Å² (4u7e, CE
  missing, so nothing covers NZ). These are model artifacts, not biology; the ≥ 60 bin is essentially empty.
- Structures failed: **9i1b** (faulty assembly annotation → 23 pairs of chain copies at identical coordinates →
  FreeSASA cannot compute overlapping atoms; correctly rejected). Ten huge assemblies (108k–624k residues) needed the
  extended labels of section 6 and were computed afterwards.

## 6. Keeping chain identity through FreeSASA

**The problem.** FreeSASA (both the Python bindings and the CLI, v2.1.2) keeps only the **first character** of a chain
name: `A1`, `A2`, `AA`, `AAA` all become `A`, and residues can even be listed under the wrong chain in the JSON.

**The solution.** Chain names are not used at all. Before running FreeSASA every residue of the assembly gets a
**unique residue label** (1…99,999, then −999…9999 with insertion codes A–Z, a–z: 671,947 labels, all within
FreeSASA's 5-character limit). A lookup table maps each label back to its biological chain (`A1`), author chain,
residue number and UniProt position. `validate` requires every label to come back exactly once with the right
residue name, otherwise the structure fails rather than being saved.

**Evidence it works**
- NZ areas recomputed independently (atom by atom with our own labels): max difference **0.0000 Å²** (236 values).
- Contacts join to SASA copies by `(structure, bio_chain, resnum, icode)`: **100%** of 30,675 lysine contact sides in
  25 random structures; 40/40 with the same UniProt position in 1wh9.

Why not unique one-character chain names? Only ~62 safe characters, and ~0.9% of assemblies have more author chains
than that (up to 84); residue labels have no such limit.

## 7. Contacts: definition and classes

**A contact** = the lysine **NZ** within **6.5 Å** of an atom of another **polymer** residue (protein or nucleic acid;
ligands and water excluded), **excluding sequence-adjacent residues**, found in the biological assembly.
Modified lysines (MLY, ALY, KCX, …) count as lysines for contacts.

**Inter- vs intra-chain.** Inter-chain = the two residues are on **different biological chain copies** — this includes
two copies of the same protein in a homo-oligomer. Intra-chain = same chain copy.

**Contact classes** (a lysine copy is in a class if one of its contacts satisfies it)
| class | criterion |
|---|---|
| `dist4.5` | NZ within 4.5 Å of the partner atom |
| `salt_bridge` | NZ to carboxylate O ≤ 4.0 Å |
| `hydrogen_bond` | ≤ 3.5 Å donor–acceptor |
| `cation_pi` | NZ to aromatic ring ≤ 6.0 Å, angle ≤ 40° |
| `covalent_isopeptide` | ≤ 1.6 Å (covalent) — essentially empty: 8 inter, 6 intra, 0 AlphaFold lysines |
| `any_bond` | any of the four bond types |

The 6.5 Å distance class was dropped as too permissive: ~90% of lysine copies have *some* intra-chain atom within
6.5 Å of NZ.

**Sources:** PDB inter-chain, PDB intra-chain, AlphaFold intra-chain (the monomer has no partner chains).

**Why chain copies matter.** Previously a lysine's multimer SASA was aggregated over **all** copies in all
structures, whether or not that copy was at the interface. Now the inter-chain SASA uses **only the copies that
actually have the inter-chain contact**. Effect: median `asa_nz` of inter-chain lysines is 21.1 Å² over their contact
copies vs 31.5 Å² over all copies.

## 8. Aggregation: copies, lysines, peptides

### Contact-class columns (used by eval6, eval7, eval8)
The value of a group is computed **only from the chain copies where the lysine has that contact**.
Example: the **inter-chain salt-bridge** `asa_nz` of a lysine uses only the chain copies (in any PDB structure) in
which *that copy* of the lysine forms an inter-chain salt bridge. Copies of the same lysine without that contact —
in other structures, or other copies in the same assembly — are **not** used.

**Level 1 — chain copy.** One `asa_nz` per (structure, biological chain copy, lysine), from the multimer SASA of that
exact copy. A copy with several contacts of the class counts **once**.

**Level 2 — lysine.** Over the copies **in the class** only: `min`, `mean`, `max` and the number of such copies
(`asa_nz_pdb_{inter|intra}_{class}_{min,mean,max}`, `n_pdb_{inter|intra}_{class}`).
- No contact PDB: every copy qualifies (none has a contact within 4.5 Å), so all copies are used.
- AlphaFold groups: a single value per lysine (the monomer), no copies.

**Level 3 — peptide.** Over the peptide's lysines **that are in the class** (lysines not in the class are not part of
that group):
| name | computed as | intuition |
|---|---|---|
| **max** (`…_max_max`) | max over the class copies, then max over the class lysines | the most exposed state seen at that interface |
| **min** (`…_min_min`) | min over the class copies, then min over the class lysines | the most buried state seen at that interface |
| **avg** (`…_mean_avg`) | mean over the class copies per lysine, then mean over the class lysines | a typical value; each lysine weighted equally regardless of its number of copies |

Effect of restricting to contact copies: for inter-chain lysines the median `asa_nz` is 21.1 Å² over their contact
copies vs 31.5 Å² over all their copies.

### Older all-copies columns (kept, class-independent)
`asa_nz_multimer_{min,mean,max}` (per lysine) and their peptide aggregates `asa_nz_multimer_min_min`,
`asa_nz_multimer_max_max`, `asa_nz_multimer_mean_{avg,min,max}` use **all** chain copies of the lysine in all
structures, whether or not the copy has any contact. They are not used by the current notebooks (eval6 / eval7 /
eval8 use the contact-class columns above); the retired `obsolete/eval4.ipynb` used them for its Inter group.

### Missing values
- Contact-class columns: aggregated over the lysines in the class; other lysines are ignored.
- Older columns (monomer `asa_nz_{avg,min,max}`, `asa_nz_multimer_*`, B-factors): **every** lysine of the peptide
  must have a value, otherwise the peptide value is NaN.

### Peptide flags
A peptide is in a contact class if **any** of its lysines is. It is **no contact** only if **all** its lysines are.
**Purely intra** is decided at the peptide level: an intra-chain class and **no** lysine with an inter-chain bond
(`pdb_inter_any_bond`).

### Caveat on max / min over many structures
The more structures (and copies in the class) a protein has, the more extreme its max and min become. Max-of-max and
min-of-min are therefore biased toward the extremes for well-studied proteins; avg is the more robust summary.

## 9. No contact: definition, exhaustiveness, overlaps, coverage

**Definition.** No contact = NZ has **no** atom of a non-adjacent polymer residue within **4.5 Å** in **any** chain copy
of **any** structure of that source (`no_contact_pdb`), or in the AlphaFold model (`no_contact_af`). Its SASA: PDB,
multimer over all copies (they all qualify); AlphaFold, the monomer.

**Can no contact overlap with the distance classes?** No, by construction — verified: 0 overlap at the lysine and
peptide level, within the same source.

**Two expected exceptions**
- **Cation-π:** allowed up to 6.0 Å, so a lysine with nothing within 4.5 Å can still have a cation-π contact at
  4.5–6 Å (10 PDB lysines, 47 AlphaFold lysines). Salt bridges and H-bonds are shorter, so never overlap.
- **Across sources:** a lysine can be at a PDB interface and contact-free in the monomer (1,326 lysines with a PDB
  inter-chain bond are no contact in AlphaFold) — the monomer has no partner chains.

**Are inter / intra / no contact exhaustive?** Only for peptides **covered** by that source. In the Venn diagrams
(2,756 unique peptides), 1,330 are in none of the PDB sets:
- 1,326: no lysine of the peptide is in any PDB structure (with an NZ):
  - 618 peptides (449 proteins): the protein has **no PDB structure**;
  - 708 peptides (476 proteins): the protein has structures but they do not include these lysines (unmodelled
    termini/loops, constructs, side chains truncated before NZ, or chains failing the UniProt alignment).
- 1: mixed coverage, e.g. `MAPKGSSKQQSEEDLLLQDFSR` (Q9UNL2): K4 lies outside every structure (models cover residues
  7–185), K8 is covered and contact-free → the peptide cannot be called no contact (all lysines required).
- 3: proteins with structures but **no saved PDB contacts file** (tubulins Q71U36, P68371) → contact flags NA.

For AlphaFold only 80 peptides are uncovered (no model, or lysine missing from it).

## 10. Monomer vs multimer: what is (never) mixed

**Rule:** a number is never computed from both AlphaFold and PDB. Every column comes from one source.
**Plots may place groups from different sources side by side** (e.g. Inter PDB next to Intra AF), with titles saying which.

- **Inter PDB** always uses the multimer (only PDB has partner chains).
- `eval7_monomer`: Intra AF and No contact AF, both membership and `asa_nz` from AlphaFold.
- `eval7_multimer`: Intra PDB and No contact PDB, both membership and `asa_nz` from PDB chain copies.
- Why the difference matters: the monomer lacks partners, so its accessibility is systematically higher for
  interface lysines; multimer values reflect the complexes that were crystallized, which can include artificial
  lattice-like or construct-dependent contacts.

## 11. B-factors

- From the **deposited** PDB model (asymmetric unit), per author chain. Assembly copies of a chain share B-factors.
- **z-scores** normalized over all protein heavy atoms of the **same chain** (0 = chain average, positive = more
  mobile). Raw B-factors are not comparable across structures (resolution, refinement).
- Per lysine: NZ z-score min/mean/max over chains; residue-average z-score only as the **mean** over structures.
- Per peptide: NZ max-of-max, NZ min-of-min, residue mean avg/min/max.
- **Class-independent:** the value does not depend on the contact class; it is only plotted per group.
- Correlations with MS_DSA are weak (e.g. NZ max, version A, Inter: 0.02–0.20 across runs).

## 12. Plots and statistics

- **Histograms:** each group's values are binned and converted to **percent within the group**, so groups of very
  different sizes can be compared. Values outside the bin edges are reported in the legend.
- **All-runs histograms (eval7):** per run percentages, then the **mean over runs**; error bars = standard deviation
  over runs; legends give peptides summed over runs.
- **All-runs scatter:** all runs on one plot, not distinguished.
- **Same peptides across panels:** structural histograms count only peptides with a value in the MS_DSA column used
  (`require=dsa_column`), so counts match the scatter plots and correlations.
- **Correlation:** Spearman (rank-based; MS_DSA is bounded and piles up at 1).
- **Venn diagrams (eval8):** unique peptides with a shared-charge MS_DSA in some run (flags do not depend on the run).
- **Bins:** MS_DSA `0–0.1`, three equal bins over 0.1–0.9, three over 0.9–1; asa_nz width 10 with an open `≥60`;
  B-factor z width 1 from −3 to 5 with open ends.
- **Group colours:** Inter PDB red, Intra PDB blue, Intra AF green, No contact grey.

**Notebooks:** eval6 (contact classes, per run), eval7_monomer / eval7_multimer (all runs combined), eval8_venn
(group overlaps). Retired to `obsolete/`: eval3 (original all-charge MS_DSA incl. missing channels), eval4
(shared-charge MS_DSA with all-copies multimer asa_nz), eval5 (B-factors with the old group definitions) — their
Inter group combined an inter-chain flag and a salt-bridge flag that need not refer to the same contact.

## 13. Sanity checks done (question → answer)

| question | answer |
|---|---|
| Does FreeSASA lose chain identity? | Its chain names do (1 character), so they are not used: unique residue labels map every residue back to its biological chain; verified 0.0000 Å² difference and 100% join with contacts. |
| Are `asa_nz` values up to 81 real NZ values? | Yes, but artifacts: 16 lysines > 60 Å², from stretched CE–NZ bonds or a missing CE (4u7e). |
| Does `--separate-chains` give complex accessibility? | No — it computes each chain alone; we run the whole assembly. |
| Is the inter-chain SASA always from the multimer? | Yes, in every plot. |
| Is the same NZ in different copies averaged together? | No: one value per copy; aggregation is explicit (min/mean/max). Symmetric copies can have identical values legitimately (e.g. 2e1a A1/A2). |
| Which MS_DSA and which peptides are in each plot? | Stated in every title and flagged in the run tables; versions A/B defined in section 2. |
| Are peptides with a missing channel included? | Not in the current notebooks (shared-charge MS_DSA is NaN without both channels); flagged by `channel_absent`. The retired eval3 showed the all-charge MS_DSA including them. |
| Does ComputeManager terminate with failures? | Fixed: failed keys are excluded after a refresh. |
| Can no contact overlap distance classes? | No (0 overlap within a source); cation-π and cross-source overlaps are expected. |
| Why are some peptides in none of inter/intra/no contact? | Coverage: no PDB structure (618), structures without these lysines (708), mixed coverage (1), missing contacts files (3). |
| Is the pipeline per run? | Yes: MS_DSA and all flags are per (peptide, run). |
| Were existing results changed by the new columns? | No: all previous columns identical value for value after every rebuild. |

## 14. Anticipated questions

1. **Why NZ SASA and not residue SASA?** NZ is the labelled atom; residue SASA includes backbone and CB–CE, which
   say little about whether NZ can react.
2. **Why the biological assembly, not the asymmetric unit?** The asymmetric unit can cut real interfaces or contain
   crystal contacts; the assembly is the author/PISA-annotated functional unit. Caveat: only the first assembly.
3. **Are crystal contacts counted?** Only if they are inside the biological assembly; symmetry mates outside it are
   not (`image_idx == 0`).
4. **Homo-oligomer contacts?** Counted as inter-chain (different chain copies of the same protein).
5. **Why remove ligands and nucleic acids from SASA?** To measure burial by proteins, consistent with the protein
   contacts; a lysine bound to DNA may therefore look accessible.
6. **Why several structures per protein?** Proteins are crystallized in different complexes and states; min/max
   capture the range, avg a typical value. More structures → more extreme min/max.
7. **Why 4.5 Å for no contact?** 6.5 Å marks ~90% of lysine copies as in contact; 4.5 Å is closer to direct contact.
8. **Why are MS_DSA values piled at 1?** Many peptides are mostly light (accessible); with zero heavy area MS_DSA = 1
   exactly (version A). Version B removes those.
9. **Why the shared-charge MS_DSA?** Unequal charge states across channels bias the sum toward light.
10. **Isn't AlphaFold a prediction?** Yes; it is kept separate, and low-confidence regions (e.g. disordered tails)
    may look more exposed or more compact than reality. pLDDT is not used as a filter here.
11. **Why do so few peptides remain?** Shared-charge MS_DSA needs both channels (version A keeps ≈ 11–16% of the peptide rows per run), and
    PDB classes need structural coverage of the lysine.
12. **Peptides shared between proteins?** Split into one row per protein; the same MS_DSA appears for each.

## 15. Known limitations and open items

- Only the first biological assembly per structure; alternative assemblies are ignored.
- Altlocs: the first conformer is kept for SASA.
- Modified lysines (MLY, ALY, KCX, …) count as lysines for contacts but the SASA lookup uses `LYS` only; this is a
  likely (not verified) reason why 0.0–0.2% of contact copies per class find no SASA row.
- No saved PDB contacts for two large proteins (Q71U36, P68371); 9i1b has no SASA.
- 10 SIFTS-only structures: residues with a missing SIFTS begin position are not mapped (e.g. 6obj).
- Peptide location = first exact match; repeats within a protein are not disambiguated.
- `purely_intra_af` treats proteins without any PDB structure as "not inter-chain".
- Max/min depend on the number of structures available per protein (section 8).
- Contacts ignore ligands and water, so "no contact" lysines may still bind small molecules.

## 16. Live checks of key numbers (recomputed from the current files)

In [2]:
import pandas as pd
from dsa.ms_dsa.ms_dsa_v2 import MS_DSA_Builder
from dsa.ms_dsa.evaluation1.ms_dsa_annotator import MSDSAAnnotator
from dsa.ms_dsa.evaluation1.lysine_properties import LysinePropertyTable, CONTACT_CLASSES

ms = MS_DSA_Builder().ms_dsa
print(f"peptide-run groups: {len(ms):,}")
print(f"  one channel absent: {ms.channel_absent.mean():.1%} | both present, a zero area: {ms.zero_area.mean():.1%} | "
      f"both present and > 0: {(~ms.channel_absent & ~ms.zero_area).mean():.1%}")
print(f"  more light than heavy charge states: {int(((ms.n_light_charges > ms.n_heavy_charges) & ~ms.channel_absent).sum())} | "
      f"more heavy: {int(((ms.n_heavy_charges > ms.n_light_charges) & ~ms.channel_absent).sum())}")

peptide-run groups: 54,760
  one channel absent: 86.4% | both present, a zero area: 7.7% | both present and > 0: 5.9%
  more light than heavy charge states: 1482 | more heavy: 15


In [3]:
lysines = LysinePropertyTable.load().table
flag = lambda c: lysines[c].fillna(False).astype(bool)
print(f"lysines: {len(lysines):,}")
pd.DataFrame({source: {c: int(flag(f"{source}_{c}").sum()) for c in CONTACT_CLASSES}
              for source in ["pdb_inter", "pdb_intra", "af_intra"]})

lysines: 25,743


,pdb_inter,pdb_intra,af_intra
dist4.5,3955,10024,12812
salt_bridge,1261,4839,6614
hydrogen_bond,2522,6939,8621
cation_pi,461,2019,904
covalent_isopeptide,8,6,0
any_bond,2682,7513,9110


In [4]:
print("no contact overlaps (lysines):")
for classes, no_contact in [("pdb_inter", "no_contact_pdb"), ("pdb_intra", "no_contact_pdb"), ("af_intra", "no_contact_af")]:
    print(f"  {classes} dist4.5 & {no_contact}: {int((flag(classes + '_dist4.5') & flag(no_contact)).sum())} | "
          f"cation_pi & {no_contact}: {int((flag(classes + '_cation_pi') & flag(no_contact)).sum())}")
print(f"asa_nz multimer max > 60 A^2: {int((lysines.asa_nz_multimer_max > 60).sum())} lysines")
inter = lysines[flag("pdb_inter_any_bond") & (lysines.n_pdb_inter_any_bond > 0)]
print(f"inter-chain lysines: median asa_nz over contact copies {inter.asa_nz_pdb_inter_any_bond_mean.median():.1f} "
      f"vs all copies {inter.asa_nz_multimer_mean.median():.1f}")

no contact overlaps (lysines):
  pdb_inter dist4.5 & no_contact_pdb: 0 | cation_pi & no_contact_pdb: 0
  pdb_intra dist4.5 & no_contact_pdb: 0 | cation_pi & no_contact_pdb: 10
  af_intra dist4.5 & no_contact_af: 0 | cation_pi & no_contact_af: 47
asa_nz multimer max > 60 A^2: 16 lysines
inter-chain lysines: median asa_nz over contact copies 21.1 vs all copies 31.5


In [5]:
runs = {r: MSDSAAnnotator.load_run(r) for r in range(5)}
for version, exclude in {"A": ["channel_absent", "no_shared_charges"],
                         "B": ["channel_absent", "no_shared_charges", "zero_area_shared"]}.items():
    counts = {}
    for r, df in runs.items():
        keep = df.MS_DSA_shared_charges.notna() & df.peptide_mapped
        for f in exclude:
            keep &= ~df[f]
        counts[r] = int(keep.sum())
    print(f"version {version} peptides per run: {counts}")

version A peptides per run: {0: 1580, 1: 1456, 2: 1361, 3: 1455, 4: 1840}
version B peptides per run: {0: 742, 1: 668, 2: 675, 3: 721, 4: 784}
